In [1]:
import ee
import multiprocessing
from tqdm import tqdm
from retry import retry
import requests

In [2]:
ee.Authenticate()
ee.Initialize(project='ee-kerrilu')
ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com')

In [11]:
# https://gorelick.medium.com/fast-er-downloads-a2abd512aa26

rectangle = ee.Geometry.Polygon([[[-125.25859375, 49.5120116829952],
          [-125.25859375, 24.748325329747296],
          [-66.54765625000002, 24.748325329747296],
          [-66.54765625000002, 49.5120116829952]]]) # rectangle around contiguous US
LSIB = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
US = LSIB.filter(ee.Filter.eq('country_na', 'United States')).filterBounds(rectangle).geometry()
grid = US.coveringGrid('EPSG:5070', 4096)

NAIP = ee.ImageCollection("USDA/NAIP/DOQQ")
RGB = NAIP.filterDate('2021-01-01', '2024-01-01')
calibration_points = ee.FeatureCollection("projects/ee-kerrilu/assets/US-fields/calibration_points")
calibration_points = calibration_points.filter(ee.Filter.lte('point_id', 1200)).sort('point_id')

@retry(tries=10, delay=1, backoff=2)
def getResult(enumerated_point):
    i, point = enumerated_point
    coord = ee.Geometry.Point(point['coordinates'])
    tile = grid.filterBounds(coord).geometry() # 512x512 tile containing the point
    final_tile = grid.filterBounds(tile.buffer(10, ee.ErrorMargin(0.001, 'projected'), 'EPSG:5070')).geometry() \
                    .bounds(ee.ErrorMargin(0.001, 'projected'), 'EPSG:5070') # nine 512x512 tiles surrounding the point
    images = RGB.filterBounds(final_tile).sort('system:time_start')
    image_mosaic = images.mosaic()
    image = image_mosaic.clip(final_tile)
    image_with_sampled_point = image.paint(featureCollection=ee.FeatureCollection([ee.Feature(coord)]), color=300)

    filename = f'calibration_data/images/{(i+1):04d}.tif'
    url = image.getDownloadURL({
        'dimensions':'1536x1536',
        'crs':'EPSG:5070',
        'format': 'GEO_TIFF'
    })
    r = requests.get(url, stream=True)
    if r.status_code != 200:
        raise r.raise_for_status()
    with open(filename, 'wb') as out_file:
        out_file.write(r.content)

    filename = f'calibration_data/images_with_sampled_point/{(i+1):04d}.tif'
    url = image_with_sampled_point.getDownloadURL({
        'dimensions':'1536x1536',
        'crs':'EPSG:5070',
        'format': 'GEO_TIFF'
    })
    r = requests.get(url, stream=True)
    if r.status_code != 200:
        raise r.raise_for_status()
    with open(filename, 'wb') as out_file:
        out_file.write(r.content)

# DOWNLOAD IMAGES 
points = calibration_points.aggregate_array('.geo').getInfo()
with multiprocessing.Pool(25) as pool:
    results = list(tqdm(pool.imap(getResult, enumerate(points)), total=len(points)))
    pool.close()
    pool.join()

100%|██████████| 1200/1200 [21:18<00:00,  1.07s/it]


# Fix the points that have fewer than 9 surrounding tiles (points at the edge of the US)

In [26]:
# Find the points that have fewer than 9 surrounding tiles

rectangle = ee.Geometry.Polygon([[[-125.25859375, 49.5120116829952],
          [-125.25859375, 24.748325329747296],
          [-66.54765625000002, 24.748325329747296],
          [-66.54765625000002, 49.5120116829952]]]) # rectangle around contiguous US
LSIB = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
US = LSIB.filter(ee.Filter.eq('country_na', 'United States')).filterBounds(rectangle).geometry()
grid = US.coveringGrid('EPSG:5070', 4096)

NAIP = ee.ImageCollection("USDA/NAIP/DOQQ")
RGB = NAIP.filterDate('2021-01-01', '2024-01-01')
calibration_points = ee.FeatureCollection("projects/ee-kerrilu/assets/US-fields/calibration_points")
calibration_points = calibration_points.filter(ee.Filter.lte('point_id', 1200)).sort('point_id')

@retry(tries=10, delay=1, backoff=2)
def getResult(enumerated_point):
    i, point = enumerated_point
    coord = ee.Geometry.Point(point['coordinates'])
    tile = grid.filterBounds(coord).geometry() # 512x512 tile containing the point
    surrounding_tiles = grid.filterBounds(tile.buffer(10, ee.ErrorMargin(0.001, 'projected'), 'EPSG:5070'))
    if surrounding_tiles.size().getInfo() != 9:
        return (i, point)

points = calibration_points.aggregate_array('.geo').getInfo()
with multiprocessing.Pool(25) as pool:
    enumerated_points_with_tile_error = list(tqdm(pool.imap(getResult, enumerate(points)), total=len(points)))
    pool.close()
    pool.join()

100%|██████████| 1200/1200 [02:32<00:00,  7.89it/s]


In [28]:
enumerated_points_with_tile_error = [x for x in enumerated_points_with_tile_error if x is not None]
print(len(enumerated_points_with_tile_error))
print(enumerated_points_with_tile_error)

7
[(116, {'type': 'Point', 'coordinates': [-121.78320449042472, 36.747844182340195]}), (219, {'type': 'Point', 'coordinates': [-75.86088920455019, 36.490183946168926]}), (421, {'type': 'Point', 'coordinates': [-102.93238600068966, 48.99172411871084]}), (423, {'type': 'Point', 'coordinates': [-103.17784167727763, 48.96845207031291]}), (576, {'type': 'Point', 'coordinates': [-78.3351106060757, 43.35996867022977]}), (963, {'type': 'Point', 'coordinates': [-80.35812449020514, 41.99979123949295]}), (1056, {'type': 'Point', 'coordinates': [-121.7657203551942, 36.8061558468161]})]


In [30]:
# Re-export the images for these points using the larger grid

rectangle = ee.Geometry.Polygon([[[-125.25859375, 49.5120116829952],
          [-125.25859375, 24.748325329747296],
          [-66.54765625000002, 24.748325329747296],
          [-66.54765625000002, 49.5120116829952]]]) # rectangle around contiguous US
grid = rectangle.coveringGrid('EPSG:5070', 4096)

NAIP = ee.ImageCollection("USDA/NAIP/DOQQ")
RGB = NAIP.filterDate('2021-01-01', '2024-01-01')
calibration_points = ee.FeatureCollection("projects/ee-kerrilu/assets/US-fields/calibration_points")
calibration_points = calibration_points.filter(ee.Filter.lte('point_id', 1200)).sort('point_id')

@retry(tries=10, delay=1, backoff=2)
def getResult(enumerated_point):
    i, point = enumerated_point
    coord = ee.Geometry.Point(point['coordinates'])
    tile = grid.filterBounds(coord).geometry() # 512x512 tile containing the point
    '''
    surrounding_tiles = grid.filterBounds(tile.buffer(10, ee.ErrorMargin(0.001, 'projected'), 'EPSG:5070'))
    if surrounding_tiles.size().getInfo() == 9:
        print("correct number of tiles")
    '''
    final_tile = grid.filterBounds(tile.buffer(10, ee.ErrorMargin(0.001, 'projected'), 'EPSG:5070')).geometry() \
                    .bounds(ee.ErrorMargin(0.001, 'projected'), 'EPSG:5070') # nine 512x512 tiles surrounding the point
    images = RGB.filterBounds(final_tile).sort('system:time_start')
    image_mosaic = images.mosaic()
    image = image_mosaic.clip(final_tile)
    image_with_sampled_point = image.paint(featureCollection=ee.FeatureCollection([ee.Feature(coord)]), color=300)

    filename = f'calibration_data/images/{(i+1):04d}.tif'
    url = image.getDownloadURL({
        'dimensions':'1536x1536',
        'crs':'EPSG:5070',
        'format': 'GEO_TIFF'
    })
    r = requests.get(url, stream=True)
    if r.status_code != 200:
        raise r.raise_for_status()
    with open(filename, 'wb') as out_file:
        out_file.write(r.content)

    filename = f'calibration_data/images_with_sampled_point/{(i+1):04d}.tif'
    url = image_with_sampled_point.getDownloadURL({
        'dimensions':'1536x1536',
        'crs':'EPSG:5070',
        'format': 'GEO_TIFF'
    })
    r = requests.get(url, stream=True)
    if r.status_code != 200:
        raise r.raise_for_status()
    with open(filename, 'wb') as out_file:
        out_file.write(r.content)

# DOWNLOAD IMAGES 
with multiprocessing.Pool(25) as pool:
    results = list(tqdm(pool.imap(getResult, enumerated_points_with_tile_error), total=len(enumerated_points_with_tile_error)))
    pool.close()
    pool.join()

100%|██████████| 7/7 [00:29<00:00,  4.20s/it]


# Download a cloudless version of image 0399 (for labeling only; during calibration, the model is run on the image with clouds)

In [5]:
# https://gorelick.medium.com/fast-er-downloads-a2abd512aa26

rectangle = ee.Geometry.Polygon([[[-125.25859375, 49.5120116829952],
          [-125.25859375, 24.748325329747296],
          [-66.54765625000002, 24.748325329747296],
          [-66.54765625000002, 49.5120116829952]]]) # rectangle around contiguous US
LSIB = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
US = LSIB.filter(ee.Filter.eq('country_na', 'United States')).filterBounds(rectangle).geometry()
grid = US.coveringGrid('EPSG:5070', 4096)

NAIP = ee.ImageCollection("USDA/NAIP/DOQQ")
RGB = NAIP.filterDate('2021-01-01', '2024-01-01')
calibration_points = ee.FeatureCollection("projects/ee-kerrilu/assets/US-fields/calibration_points")
calibration_points = calibration_points.filter(ee.Filter.lte('point_id', 1200)).sort('point_id')

def getResult(enumerated_point):
    i, point = enumerated_point
    coord = ee.Geometry.Point(point['coordinates'])
    tile = grid.filterBounds(coord).geometry() # 512x512 tile containing the point
    final_tile = grid.filterBounds(tile.buffer(10, ee.ErrorMargin(0.001, 'projected'), 'EPSG:5070')).geometry() \
                    .bounds(ee.ErrorMargin(0.001, 'projected'), 'EPSG:5070') # nine 512x512 tiles surrounding the point
    images = RGB.filterBounds(final_tile).sort('system:time_start', False)
    image_mosaic = images.mosaic()
    image = image_mosaic.clip(final_tile)
    image_with_sampled_point = image.paint(featureCollection=ee.FeatureCollection([ee.Feature(coord)]), color=300)

    filename = f'calibration_data/{(i+1):04d}_cloudless.tif'
    url = image.getDownloadURL({
        'dimensions':'1536x1536',
        'crs':'EPSG:5070',
        'format': 'GEO_TIFF'
    })
    r = requests.get(url, stream=True)
    if r.status_code != 200:
        raise r.raise_for_status()
    with open(filename, 'wb') as out_file:
        out_file.write(r.content)

# DOWNLOAD IMAGES 
points = calibration_points.aggregate_array('.geo').getInfo()
getResult((398, points[398]))